# YOLO (detection)

**Phase:** 1 | **Model:** YOLO26-S (COCO pretrained) | **Data:** `bench/<mode>/` from `02_data_prep` | **Settings:** [configs/yolo26s.yaml](../../configs/yolo26s.yaml)

Trains YOLO to draw boxes around defects. For the benchmark an image counts as defective if any box passes the confidence threshold, and its score is its highest box confidence.

**Why YOLO26-S.** Several runs were tried and the small model gave the best results so far. It trains with light built-in augmentation (brightness, small shift and zoom, no flips) and stops early after 7 epochs without improvement.

Plan: `gray` first, then a second full run on `color` (change `MODE`). Outputs go to `results/yolo/<dataset>/<run name>/`; checkpoints are git-ignored.

## 1. Setup

In [1]:
import os

import pandas as pd
import torch
import yaml
from IPython.display import Image, display
from ultralytics import YOLO

from vision_benchmark.evaluation import apply_conf, best_threshold, detection_summary, image_metrics, recall_by_type
from vision_benchmark.models import predict_yolo
from vision_benchmark.utils import show_detections

In [2]:
DATASET = "pcb1"
MODE = "gray"   # "gray" first, then "color"
MODEL_NAME = "yolo26s"

ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
BENCH = os.path.join(ROOT, "bench")
CFG_PATH = os.path.join(ROOT, "configs", f"{MODEL_NAME}.yaml")
DATA_YAML = os.path.join(BENCH, MODE, "data.yaml")

# must be an absolute path, otherwise Ultralytics saves into its own global runs/ folder
RUNS_DIR = os.path.join(ROOT, "results", "yolo", DATASET)
RUN_NAME = f"{MODEL_NAME}_{MODE}"
RUN_DIR = os.path.join(RUNS_DIR, RUN_NAME)

print("data:", DATA_YAML)
print("run :", RUN_DIR)

data: D:\Full AI\Projects\GitHub - Kaggle Projects\vision-model-selection-benchmark\bench\gray\data.yaml
run : D:\Full AI\Projects\GitHub - Kaggle Projects\vision-model-selection-benchmark\results\yolo\pcb1\yolo26s_gray


## 2. Config

All training settings live in the yaml file so every run is documented in one place.

In [3]:
with open(CFG_PATH) as f:
    cfg = yaml.safe_load(f)

CFG = cfg["train"]
# pretrained weights live in <repo>/weights/, not in the notebook folder
WEIGHTS = os.path.normpath(os.path.join(ROOT, cfg["weights"]))
print("weights:", WEIGHTS)
pd.Series(CFG, name="value").to_frame()

weights: D:\Full AI\Projects\GitHub - Kaggle Projects\vision-model-selection-benchmark\weights\yolo26s.pt


,value
imgsz,1024
epochs,50
patience,7
batch,16
nbs,16
optimizer,AdamW
lr0,0.001
lrf,0.1
device,0
amp,True


## 3. GPU check

In [4]:
assert torch.cuda.is_available(), "No CUDA GPU found"

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}:", props.name, "|", round(props.total_memory / 2**30, 1), "GB")
print("torch", torch.__version__, "| CUDA", torch.version.cuda)

GPU 0: NVIDIA GeForce RTX 2060 SUPER | 8.0 GB
torch 2.11.0+cu130 | CUDA 13.0


## 4. Model

If the weights file is missing, Ultralytics downloads it to the exact path we give.

In [5]:
model = YOLO(WEIGHTS)
model.info()

YOLO26s summary: 260 layers, 10,009,784 parameters, 0 gradients, 23.1 GFLOPs


(260, 10009784, 0, 23.0793216)

## 5. Train

Each epoch prints the train losses, then precision, recall and mAP on val. Training stops early when val has not improved for `patience` epochs. `weights/best.pt` is the best val epoch.

In [ ]:
results = model.train(data=DATA_YAML, project=RUNS_DIR, name=RUN_NAME, **CFG)

## 6. Training results

Files saved by the run:

In [ ]:
sorted(os.listdir(RUN_DIR))

Metrics per epoch (last 5):

In [ ]:
history = pd.read_csv(os.path.join(RUN_DIR, "results.csv"))
history.tail()

Loss and metric curves:

In [ ]:
display(Image(filename=os.path.join(RUN_DIR, "results.png")))

Box-level precision-recall curve and confusion matrix on val (Ultralytics counts boxes, not images):

In [ ]:
display(Image(filename=os.path.join(RUN_DIR, "BoxPR_curve.png")))

In [ ]:
display(Image(filename=os.path.join(RUN_DIR, "confusion_matrix.png")))

A train batch as the model sees it, now with the light augmentation (small shifts, zoom, brightness; still no flips):

In [ ]:
display(Image(filename=os.path.join(RUN_DIR, "train_batch0.jpg")))

## 7. Image-level results on val

The benchmark question is per image: does this board have a defect? `predict_yolo` runs the best checkpoint once on every image and keeps all boxes above 0.01, so any confidence threshold can be tried afterwards without running the model again.

In [ ]:
best = YOLO(os.path.join(RUN_DIR, "weights", "best.pt"))
split_df = pd.read_csv(os.path.join(BENCH, "split.csv"))

val_pred = predict_yolo(best, split_df[split_df["split"] == "val"], BENCH, MODE, imgsz=CFG["imgsz"])
val_pred[["image_id", "label", "defect_type", "score"]].sort_values("score", ascending=False).head(10)

Threshold-free scores (AUROC, AP) and the confidence with the best F1 on val:

In [ ]:
VAL_CONF = best_threshold(val_pred["label"], val_pred["score"])
pd.Series(image_metrics(val_pred["label"], val_pred["score"], VAL_CONF), name="val").to_frame()

**`CONF` is the confidence threshold used below. It starts at the val choice; set any value to see what changes.**

In [ ]:
CONF = VAL_CONF   # for example 0.25 or 0.5

val_eval = apply_conf(val_pred, CONF)
detection_summary(val_eval)

Found and missed defective images per defect type and for images with a small box:

In [ ]:
recall_by_type(val_eval)

Missed defects (FN). Green = true box, red = predicted box with its confidence; the right side zooms on the boxes:

In [ ]:
show_detections(val_eval, num_images=6, outcome="FN", save_plot=os.path.join(RUN_DIR, "val_missed.png"))

Found defects (TP), to compare where the predicted box lands against the true one:

In [ ]:
show_detections(val_eval, num_images=6, outcome="TP", save_plot=os.path.join(RUN_DIR, "val_found.png"))

False alarms on normal boards (FP):

In [ ]:
show_detections(val_eval, num_images=6, outcome="FP", save_plot=os.path.join(RUN_DIR, "val_false_alarms.png"))

In [ ]:
val_eval.drop(columns=["path"]).to_csv(os.path.join(RUN_DIR, "val_predictions.csv"), index=False)

## 8. Test

Same steps on test with the threshold from val. Changing `TEST_CONF` after looking at the test results makes the test score optimistic, so the number to report is the one with the val threshold.

In [ ]:
test_pred = predict_yolo(best, split_df[split_df["split"] == "test"], BENCH, MODE, imgsz=CFG["imgsz"])

TEST_CONF = CONF
pd.Series(image_metrics(test_pred["label"], test_pred["score"], TEST_CONF), name="test").to_frame()

In [ ]:
test_eval = apply_conf(test_pred, TEST_CONF)
detection_summary(test_eval)

In [ ]:
recall_by_type(test_eval)

In [ ]:
show_detections(test_eval, num_images=6, outcome="FN", save_plot=os.path.join(RUN_DIR, "test_missed.png"))

In [ ]:
show_detections(test_eval, num_images=6, outcome="TP", save_plot=os.path.join(RUN_DIR, "test_found.png"))

In [ ]:
show_detections(test_eval, num_images=6, outcome="FP", save_plot=os.path.join(RUN_DIR, "test_false_alarms.png"))

In [ ]:
test_eval.drop(columns=["path"]).to_csv(os.path.join(RUN_DIR, "test_predictions.csv"), index=False)

## Next

- Second run with `MODE = "color"`.
- Latency, VRAM and model size, then the per-image log in `results/per_image/`.